# Franka Lift — SmolVLA İnce Ayarı

Isaac Sim'de toplanan 152 bölümlük gösterim verisiyle SmolVLA modelini eğitiyoruz.

**Önce yapılacak:** Menüden `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU` seç.

**Uyarı:** Ücretsiz Colab 90 dakika işlem yapılmazsa oturumu keser, en fazla 12 saat çalışır.
Bu yüzden checkpoint'leri Drive'a kaydediyoruz — kesilirse kaldığın yerden devam edersin.


## 1. GPU kontrolü


In [ ]:
!nvidia-smi
import sys; print('Python:', sys.version)


## 2. Google Drive'ı bağla

`franka_lift_sim.tar.gz` dosyasını Drive'ında `franka_vla/` klasörüne yüklemiş olmalısın.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls -lh /content/drive/MyDrive/franka_vla/


## 3. LeRobot kurulumu

`lerobot 0.6.1` Python 3.12 istiyor. Colab'ın sürümü eskiyse hücre uyarı verip
uyumlu en yüksek sürüme düşecek.


In [ ]:
import sys
PY = sys.version_info
print(f'Colab Python: {PY.major}.{PY.minor}')
if (PY.major, PY.minor) >= (3, 12):
    VER = '0.6.1'
else:
    VER = '0.4.4'
    print('UYARI: Python < 3.12 -> lerobot 0.4.4 kullanilacak')
print('kurulacak lerobot:', VER)


In [ ]:
# Colab'da Python degiskenini kabuk komutuna gecirmek icin {} sozdizimi kullanilir
!pip install -q "lerobot[dataset,smolvla]=={VER}"


In [ ]:
import lerobot, torch
print('lerobot:', lerobot.__version__)
print('torch  :', torch.__version__)
print('cuda   :', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## 4. Veri setini aç


In [ ]:
!mkdir -p /content/data
!tar -xzf /content/drive/MyDrive/franka_vla/franka_lift_sim.tar.gz -C /content/data
!du -sh /content/data/franka_lift_sim


## 5. Veri setini doğrula

Eğitime başlamadan önce verinin düzgün yüklendiğinden emin ol.


In [ ]:
from lerobot.datasets import LeRobotDataset
ds = LeRobotDataset('franka_lift_sim', root='/content/data/franka_lift_sim')
print('bolum :', ds.num_episodes)
print('kare  :', ds.num_frames)
print('fps   :', ds.fps)
print('gorev :', list(ds.meta.tasks.index))
s = ds[0]
for k in ['observation.images.front','observation.images.wrist','observation.state','action']:
    print(f'  {k:32s} {tuple(s[k].shape)}')


## 6. Eğitim

Parametrelerin anlamı:

| Parametre | Ne işe yarar |
|---|---|
| `--policy.pretrained_path` | Sıfırdan değil, önceden eğitilmiş SmolVLA'dan başla |
| `--batch_size` | Aynı anda kaç örnek işlensin. **VRAM yetmezse önce bunu düşür** |
| `--steps` | Kaç eğitim adımı. 10.000 başlangıç için makul |
| `--save_freq` | Kaç adımda bir checkpoint. Drive'a yazıyoruz, kesintiye karşı sigorta |
| `--policy.freeze_vision_encoder` | Görü kodlayıcıyı dondur — VRAM'i ciddi düşürür |

**OOM (bellek yetmedi) hatası alırsan:** `--batch_size` değerini 4 → 2 → 1 diye düşür.


In [ ]:
OUT = '/content/drive/MyDrive/franka_vla/train01'

# DIKKAT: klasoru ONCEDEN OLUSTURMA. lerobot-train cikti klasorunu kendisi
# yaratir ve zaten varsa 'already exists' hatasi verir (eski egitimin uzerine
# yazmamak icin koydugu koruma). Bos kalmis bir klasor varsa temizliyoruz.
import os, shutil
if os.path.isdir(OUT) and not os.listdir(OUT):
    shutil.rmtree(OUT)
    print('bos klasor silindi')
os.makedirs(os.path.dirname(OUT), exist_ok=True)   # sadece UST klasor
print('cikti yolu :', OUT)
print('zaten var  :', os.path.isdir(OUT))


In [ ]:
!lerobot-train \
  --dataset.repo_id=franka_lift_sim \
  --dataset.root=/content/data/franka_lift_sim \
  --policy.type=smolvla \
  --policy.pretrained_path=lerobot/smolvla_base \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --policy.freeze_vision_encoder=true \
  --output_dir={OUT} \
  --batch_size=4 \
  --steps=10000 \
  --save_freq=500 \
  --log_freq=50 \
  --num_workers=2 \
  --wandb.enable=false


## 7. Kesilirse: kaldığın yerden devam

Oturum koparsa 1-5. hücreleri tekrar çalıştır, sonra aşağıdakini kullan.


In [ ]:
!lerobot-train \
  --config_path={OUT}/checkpoints/last/pretrained_model/train_config.json \
  --resume=true


## 8. Eğitilmiş modeli indir

Checkpoint'ler zaten Drive'da. Yerel makinene indirmek için Drive'dan çekebilirsin.


In [ ]:
!ls -lh {OUT}/checkpoints/
!du -sh {OUT}
